# Multi-Topic Fanout: Query Routed Iceberg Tables

This notebook queries the Iceberg tables populated by the transform's multi-topic routing.

**Data flow:** `events` topic → Transform routes by `table` field → `orders`, `inventory`, `customers` topics → Iceberg tables

## Initialize Spark Session

The Spark session is already initialized. Verify it's working:

In [1]:
# Spark session is pre-configured
spark

## Count Records

Check how many records were routed to each table:

In [2]:
orders_count = spark.sql("SELECT COUNT(*) as count FROM lab.redpanda.orders").first()[0]
inventory_count = spark.sql("SELECT COUNT(*) as count FROM lab.redpanda.inventory").first()[0]
customers_count = spark.sql("SELECT COUNT(*) as count FROM lab.redpanda.customers").first()[0]

print(f"Orders: {orders_count} records")
print(f"Inventory: {inventory_count} records")
print(f"Customers: {customers_count} records")
print(f"\nTotal routed: {orders_count + inventory_count + customers_count} records")

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `lab`.`redpanda`.`orders` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 30;
'Aggregate [count(1) AS count#0L]
+- 'UnresolvedRelation [lab, redpanda, orders], [], false


## Query Orders Table

In [ ]:
spark.sql("""
    SELECT order_id, amount, timestamp,
           FROM_UNIXTIME(timestamp/1000) as order_time
    FROM lab.redpanda.orders
    ORDER BY timestamp DESC
    LIMIT 10
""").show(truncate=False)

## Query Inventory Table

In [ ]:
spark.sql("""
    SELECT product_id, quantity, timestamp,
           FROM_UNIXTIME(timestamp/1000) as update_time
    FROM lab.redpanda.inventory
    ORDER BY timestamp DESC
    LIMIT 10
""").show(truncate=False)

## Query Customers Table

In [ ]:
spark.sql("""
    SELECT customer_id, name, timestamp,
           FROM_UNIXTIME(timestamp/1000) as update_time
    FROM lab.redpanda.customers
    ORDER BY timestamp DESC
    LIMIT 10
""").show(truncate=False)

## Verify Schemas

Check the table structures:

In [ ]:
print("=== Orders Schema ===")
spark.sql("DESCRIBE lab.redpanda.orders").show(truncate=False)

print("\n=== Inventory Schema ===")
spark.sql("DESCRIBE lab.redpanda.inventory").show(truncate=False)

print("\n=== Customers Schema ===")
spark.sql("DESCRIBE lab.redpanda.customers").show(truncate=False)

## Analytics: Order Statistics

In [ ]:
spark.sql("""
    SELECT 
        COUNT(*) as total_orders,
        ROUND(SUM(amount), 2) as total_sales,
        ROUND(AVG(amount), 2) as avg_order_value,
        ROUND(MIN(amount), 2) as min_order,
        ROUND(MAX(amount), 2) as max_order
    FROM lab.redpanda.orders
""").show(truncate=False)

## Analytics: Current Inventory Levels

Get the latest quantity for each product:

In [ ]:
spark.sql("""
    WITH ranked_inventory AS (
        SELECT 
            product_id,
            quantity,
            timestamp,
            ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY timestamp DESC) as rn
        FROM lab.redpanda.inventory
    )
    SELECT 
        product_id,
        quantity as current_quantity,
        FROM_UNIXTIME(timestamp/1000) as last_updated
    FROM ranked_inventory
    WHERE rn = 1
    ORDER BY product_id
""").show(truncate=False)

## Validation: Verify Routing Correctness

Each sample batch has 1 update per table, so counts should be equal:

In [ ]:
if orders_count == inventory_count == customers_count:
    print("✅ SUCCESS: All tables have equal record counts")
    print(f"   Batches processed: {orders_count}")
    print(f"   Total updates routed: {orders_count * 3}")
else:
    print("⚠️  Record counts differ:")
    print(f"   Orders: {orders_count}")
    print(f"   Inventory: {inventory_count}")
    print(f"   Customers: {customers_count}")
    print("   Note: Wait a few seconds for Iceberg commits to complete")

## Next Steps

- Produce more batches and re-run queries to see real-time updates
- Try Iceberg time-travel: `SELECT * FROM lab.redpanda.orders.history`
- Explore table snapshots and metadata